# Lab 03 — Feature Engineering for Cybersecurity
AI Cybersecurity Masterclass · Lesson 03 · DSJ THE CRITTERS

## Research question
Can defensible derived features change the quality of threat detection?

A valid feature must be available at prediction time and must not contain target information.

In [ ]:
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

# OPTIONAL — GOOGLE COLAB ONLY: clone the repo so src/ actually exists here
try:
    import google.colab
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()

# If running from the notebooks/ folder locally:
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
# OPTIONAL — GOOGLE COLAB ONLY
# Run this cell if the UNSW-NB15 CSV files are not already in DATA_DIR.

try:
    from google.colab import files

    if not (DATA_DIR / "UNSW_NB15_training-set.csv").exists():
        print(
            "Upload UNSW_NB15_training-set.csv, "
            "UNSW_NB15_testing-set.csv, and optionally UNSW_NB15_features.csv"
        )
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)

except ImportError:
    print("Not running in Colab. Put the dataset CSV files in:", DATA_DIR)

In [ ]:
import pandas as pd

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import add_safe_derived_features, drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [ ]:
baseline = random_forest_pipeline(X_train)
baseline.fit(X_train, y_train)

base_pred = baseline.predict(X_test)
base_score = baseline.predict_proba(X_test)[:, 1]

base_metrics = binary_metrics(y_test, base_pred, base_score)
display(binary_metrics_frame("Original features", base_metrics))

In [ ]:
X_train_engineered = add_safe_derived_features(X_train)
X_test_engineered = add_safe_derived_features(X_test)

new_features = sorted(
    set(X_train_engineered.columns) - set(X_train.columns)
)

print("New features:", new_features)
if new_features:
    display(X_train_engineered[new_features].head())

In [ ]:
engineered = random_forest_pipeline(X_train_engineered)
engineered.fit(X_train_engineered, y_train)

eng_pred = engineered.predict(X_test_engineered)
eng_score = engineered.predict_proba(X_test_engineered)[:, 1]

eng_metrics = binary_metrics(y_test, eng_pred, eng_score)

comparison = pd.concat([
    binary_metrics_frame("Original features", base_metrics),
    binary_metrics_frame("Engineered features", eng_metrics),
])

display(comparison.round(4))

## Research task

Design one additional feature.

Document:

1. the security hypothesis;
2. the mathematical definition;
3. whether it exists at prediction time;
4. the before/after metrics;
5. one reason the feature might fail in another environment.